## Quickstart (illustrative)
The following cells show how to load a config and instantiate networks. **Do not run heavy training in this notebook** — it is a usage example only.

In [ ]:
# Load a YAML config (illustrative):
from src.config import ExperimentConfig
cfg = ExperimentConfig.from_yaml('configs/debug.yaml')  # illustrative only
print(cfg)

In [ ]:
# Instantiate models (shapes are illustrative)
from src.model import PolicyNetwork, ValueNetwork
policy = PolicyNetwork(obs_dim=4, action_dim=2, hidden_sizes=(32,))  # illustrative
value = ValueNetwork(obs_dim=4, hidden_sizes=(32,))  # illustrative
print(policy)
print(value)

In [ ]:
# Pseudo-training loop (illustrative; do not run here)
# for ep in range(cfg.max_episodes):
#     episodes = collect_episodes(cfg.env_name, policy, num_episodes=1)
#     returns = compute_returns(episodes, cfg.gamma)
#     # assemble losses: policy + 0.5*value + cfg.entropy_coef*entropy
#     # optimizer steps ...
#     pass

## Running experiments
To run an experiment from the command line, use the provided script: 
```
python scripts/train.py --config configs/debug.yaml
```
Results are saved to `results/` and checkpoints to `results/checkpoints/`. See `REPORT.md` for suggested experiments and figures.

## Example Experiment: Entropy Sweep (illustrative)

This section shows a self-contained example of how to run a small hyperparameter
sweep (entropy coefficient) across multiple seeds. All code is **illustrative**
and **should not be executed** inside the test harness — it's a notebook-friendly
recipe you can adapt for real runs.

### Sweep (illustrative)

```python
# PSEUDO-CODE (illustrative — do not run)
seeds = [0, 1, 2]
entropy_values = [0.0, 0.01, 0.1]

for ent in entropy_values:
    for seed in seeds:
        cfg = ExperimentConfig()
        cfg.seed = seed
        cfg.entropy_coef = ent
        # write per-run config
        run_dir = Path(f"runs/entropy_{ent}/seed_{seed}")
        run_dir.mkdir(parents=True, exist_ok=True)
        cfg.to_yaml(run_dir / "config.yaml")
        # launch training script (recommended to run from terminal)
        # subprocess.run(["python", "scripts/train.py", "--config", str(run_dir / "config.yaml")])
```

### Postprocessing & plotting (illustrative)

```python
# PSEUDO-CODE (illustrative — do not run)
# import pandas as pd
# import matplotlib.pyplot as plt
# from src.utils import save_figure
#
# all_runs = []
# for ent in entropy_values:
#     for seed in seeds:
#         p = Path(f"runs/entropy_{ent}/seed_{seed}/training_log.csv")
#         if not p.exists():
#             continue
#         df = pd.read_csv(p)
#         df['entropy'] = ent
#         df['seed'] = seed
#         all_runs.append(df)
#
# df_all = pd.concat(all_runs, ignore_index=True)
# grouped = df_all.groupby(['entropy', 'episode'])['episode_return']
# mean = grouped.mean().reset_index()
# std = grouped.std().reset_index()
#
# fig, ax = plt.subplots()
# for ent in entropy_values:
#     m = mean[mean['entropy'] == ent]
#     s = std[std['entropy'] == ent]
#     ax.plot(m['episode'], m['episode_return'], label=f'ent={ent}')
#     ax.fill_between(m['episode'], m['episode_return'] - s['episode_return'], m['episode_return'] + s['episode_return'], alpha=0.2)
# ax.set_xlabel('Episode')
# ax.set_ylabel('Episode Return')
# ax.legend()
# save_figure(fig, 'pictures/entropy_sweep.png')
```

### Run a sweep from the shell (example)

```bash
for ent in 0.0 0.01 0.1; do
  for seed in 0 1 2; do
    mkdir -p runs/entropy_${ent}/seed_${seed}
    python - <<PY
from src.config import ExperimentConfig
c = ExperimentConfig()
c.seed = int(${seed})
c.entropy_coef = float(${ent})
c.to_yaml('runs/entropy_${ent}/seed_${seed}/config.yaml')
PY
    python scripts/train.py --config runs/entropy_${ent}/seed_${seed}/config.yaml &
  done
done

# Use job control or a scheduler for larger-scale runs.
```

This notebook is illustrative — for reproducible experiments, save the exact
config and git commit in `runs/<id>/` before launching jobs.